# Manuscript figures

Generates the figures of the main text (Figs. 1, 2, 3, 5) and of the Supplemental Material (Figs. S1–S6) for *Classical fouling laws as first-moment identities of additive stochastic deposition*. Fig. 4 is made by `make_theory_figures.py`; this notebook uses the same style.

Style: Helvetica (falls back to Arial) with STIX math, ink/grey palette, left/bottom spines, light grid, panel letters and no titles, every figure drawn at its printed size (single column 3.6 in, full width 7.1 in) and included at 100 %. Deposition laws always carry Okabe–Ito colours and a line style (compound Poisson blue, solid; Gamma vermillion, dashed; inverse Gaussian green, dotted); removal mechanisms are black/grey; theory is drawn as lines, Monte Carlo as open markers.

Run from `notebooks/`; outputs go to `../Manuscript` (Fig. 3 to `../Manuscript/figures`). Seeds are fixed. Total run time about 8 minutes. Requires numpy, scipy, matplotlib.

In [ ]:
from pathlib import Path
OUT = Path("../figures"); assert OUT.exists(), "run this notebook from notebooks/"
DATA = Path("../data"); assert DATA.exists(), "run this notebook from notebooks/"
# ---- shared style (matches Fig. 4 of the main text) ----
import numpy as np, matplotlib
import matplotlib.pyplot as plt
from scipy import stats

INK, INK2, GRID = "#0b0b0b", "#52514e", "#e6e5e1"   # as make_theory_figures.py (Fig. 4)
plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Helvetica", "Arial", "DejaVu Sans"],
    "mathtext.fontset": "stix",
    "font.size": 8.5, "axes.labelsize": 8.5, "xtick.labelsize": 8, "ytick.labelsize": 8,
    "legend.fontsize": 7.5, "axes.linewidth": 0.6, "axes.edgecolor": INK2,
    "axes.labelcolor": INK, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.5, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False,
    "xtick.direction": "out", "ytick.direction": "out",
    "xtick.minor.visible": True, "ytick.minor.visible": True,
    "lines.linewidth": 1.1, "pdf.fonttype": 42, "figure.dpi": 150,
    "legend.frameon": False,
})
COL1 = 3.6           # single-column width (in), as Fig. 4
COL2 = 7.10          # full width (in)
LAWS = {"compound Poisson": -1.0, "Gamma": 0.0, "inverse Gaussian": 0.5}
LS = {"compound Poisson": ":", "Gamma": "-", "inverse Gaussian": "--"}
GREY = {"compound Poisson": "#777777", "Gamma": INK, "inverse Gaussian": "#444444"}

def panel(ax, letter):
    ax.text(0.0, 1.02, f"({letter})", transform=ax.transAxes, ha="left", va="bottom", fontsize=9)

def inline(ax, x, y, s, rot=0, **kw):
    ax.text(x, y, s, rotation=rot, rotation_mode="anchor", ha=kw.pop("ha", "left"),
            va=kw.pop("va", "bottom"), fontsize=kw.pop("fontsize", 8.5), **kw)

def save(fig, name, out=None):
    out = OUT if out is None else out
    out = Path(out)
    fig.savefig(out / f"{name}.pdf", bbox_inches="tight")
    fig.savefig(out / f"{name}.png", dpi=200, bbox_inches="tight")


# Main text

## Fig. 1 — prototypical fouling-resistance trajectories (schematic)

Kern–Seaton (asymptotic) and Ebert–Panchal (threshold) after an induction period $t_i$. Analytic; replaces the TikZ version `fig_ks_ep_schematic.tex`.

*Analytic, no seed, runtime negligible. Saved to `../figures/fig_ks_ep_schematic.{pdf,png}` (not currently included in the manuscript, which uses the TikZ version; kept for reference).*


In [ ]:
# ---- Main Fig. 1: prototypical fouling-resistance trajectories (schematic) ----
LAWC = {"KS": "#0072B2", "EP": "#D55E00"}
t = np.linspace(0, 7.4, 400); ti, tau, Rinf, slope = 0.8, 1.6, 4.2, 0.62
ks = np.where(t < ti, 0.0, Rinf * (1 - np.exp(-(t - ti) / tau)))
ep = np.where(t < ti, 0.0, slope * (t - ti))
fig, ax = plt.subplots(figsize=(COL1, 2.4))
ax.grid(False)
ax.plot(t, ks, color=LAWC["KS"], lw=1.4)
ax.plot(t, ep, color=LAWC["EP"], lw=1.4, ls="--")
ax.axhline(Rinf, color=INK2, lw=0.7, ls=(0, (4, 2)))
ax.axvline(ti, color=INK2, lw=0.6, ls=":", ymax=0.08)
ax.text(7.45, Rinf, r"$R_\infty$", va="center", ha="left", fontsize=8.5, color=INK2)
inline(ax, 4.3, 3.25, "Kern–Seaton (asymptotic)", rot=4, fontsize=7.5, color=LAWC["KS"])
inline(ax, 3.9, 1.35, "Ebert–Panchal (threshold)", rot=27, fontsize=7.5, color=LAWC["EP"])
ax.set_xticks([ti]); ax.set_xticklabels([r"$t_i$"]); ax.set_yticks([])
ax.minorticks_off()
ax.set_xlim(0, 7.4); ax.set_ylim(0, 5.0)
ax.set_xlabel(r"time $t$"); ax.set_ylabel(r"fouling resistance $R_f$")
save(fig, "fig_ks_ep_schematic")


## Fig. 2 — event spectrum, mass spectrum and sample paths

The three laws of Table II at $m=1$, $c=2$, $\tau_0=1$ ($t_c=4$, $\ell=4$). (a) Compound-Poisson event-size density (simulated jumps and the exponential law of mean $\ell/2$); (b) Lévy density of the two infinite-activity laws; (c) mass spectra $w(x)$ with common mean $\ell$; (d)–(f) one path of each law with the mean $mt$ and the $\pm2\sigma$ band.

*Seed `21`. Runtime ~2 s. Saved to `../figures/fig02_event_spectrum_6panel.{pdf,png}`.*


In [ ]:
# ---- Main Fig. 2: event spectrum, mass spectrum and sample paths (Table II) ----
from scipy.special import gamma as Gfun
rng = np.random.default_rng(21)
LCOL = {"compound Poisson": "#0072B2", "Gamma": "#D55E00", "inverse Gaussian": "#009E73"}  # Okabe-Ito
LLS = {"compound Poisson": "-", "Gamma": "--", "inverse Gaussian": ":"}
m, tau0, c = 1.0, 1.0, 2.0
tc = c**2 * tau0; ell = m * tc                        # l = 4, events of CP: rate 2/tc, mean l/2
def nu(x, a):                                         # Levy density, Eq. (13), fixed by (m, l, a)
    lam = (1 - a) / ell; B = m * lam**(1 - a) / Gfun(1 - a)
    return B * x**(-1 - a) * np.exp(-lam * x)
def w(x, a):
    return x * nu(x, a) / m

fig, ax = plt.subplots(2, 3, figsize=(COL2, 4.1))
(a1, a2, a3), (a4, a5, a6) = ax
# (a) compound Poisson: normalizable event-count spectrum
jumps = rng.exponential(ell / 2, 20000)
xx = np.linspace(0, 4, 300)
a1.hist(jumps / ell, bins=40, range=(0, 4), density=True, color=LCOL["compound Poisson"], alpha=0.25, lw=0)
a1.plot(xx, 2 * np.exp(-2 * xx), color=LCOL["compound Poisson"])
a1.set_xlabel(r"event size $x/\ell$"); a1.set_ylabel(r"event-size density $\ell\,\nu(x)/r$")
a1.text(0.97, 0.95, "simulated jumps (bars)\nexponential, mean $\\ell/2$", transform=a1.transAxes,
        ha="right", va="top", fontsize=7, color=INK2)
a1.set_xlim(0, 4); a1.set_ylim(0, 2.2)
# (b) infinite activity: not normalizable
xl = np.logspace(-3, np.log10(6), 300)
for law, a in [("Gamma", 0.0), ("inverse Gaussian", 0.5)]:
    a2.loglog(xl, nu(xl * ell, a) * ell**2 / m, color=LCOL[law], ls=LLS[law], label=law)
a2.text(0.97, 0.95, r"$\nu\propto x^{-1-\alpha}$ as $x\to0$:" "\nno finite event list", transform=a2.transAxes,
        fontsize=7, color=INK2, va="top", ha="right")
a2.set_xlabel(r"event size $x/\ell$"); a2.set_ylabel(r"$\ell^2\nu(x)/m$")
a2.set_xlim(1e-3, 6); a2.set_ylim(1e-4, 3e3)
a2.legend(loc="lower left", fontsize=7, handlelength=2.2)
# (c) mass spectra share the mean l
xm = np.linspace(1e-3, 4, 400)
cp_mass = rng.exponential(ell / 2, 200000)
a3.hist(cp_mass / ell, bins=40, range=(0, 4), weights=cp_mass / cp_mass.sum() * 10,
        color=LCOL["compound Poisson"], alpha=0.25, lw=0)          # mass-weighted jumps (density in x/l)
for law, a in [("compound Poisson", -1.0), ("Gamma", 0.0), ("inverse Gaussian", 0.5)]:
    a3.plot(xm, w(xm * ell, a) * ell, color=LCOL[law], ls=LLS[law], label=law)
a3.axvline(1, color=INK2, lw=0.6, ls=":")
a3.text(1.05, 1.88, r"mean $\ell$", fontsize=7, color=INK2, va="top")
a3.set_xlabel(r"event size $x/\ell$"); a3.set_ylabel(r"mass spectrum $\ell\,w(x)$")
a3.set_xlim(0, 4); a3.set_ylim(0, 2.0)
a3.legend(loc="center right", fontsize=7, handlelength=2.2)
# (d)-(f) one path of each law
T, dt = 60.0, 0.05; tg = np.arange(0, T + dt / 2, dt)
def path(law, seed):
    r = np.random.default_rng(seed)
    n = len(tg) - 1
    if law == "Gamma":
        inc = r.gamma(dt / tc, ell, n)
    elif law == "inverse Gaussian":
        inc = r.wald(m * dt, m * dt**2 / tc, n)
    else:
        k = r.poisson(2 * dt / tc, n); inc = np.zeros(n); nz = k > 0; inc[nz] = r.gamma(k[nz], ell / 2)
    return np.concatenate([[0], np.cumsum(inc)])
band = 2 * c * m * np.sqrt(tau0 * tg)
for axp, (law, seed) in zip((a4, a5, a6), [("compound Poisson", 21), ("Gamma", 12), ("inverse Gaussian", 11)]):
    axp.fill_between(tg, m * tg - band, m * tg + band, color=INK2, alpha=0.12, lw=0)
    axp.plot(tg, m * tg, color=INK, lw=0.8, ls=(0, (4, 2)))
    axp.step(tg, path(law, seed), where="post", color=LCOL[law], lw=1.1)
    axp.text(0.04, 0.95, law, transform=axp.transAxes, ha="left", va="top", fontsize=7.5, color=LCOL[law])
    axp.set_xlim(0, T); axp.set_ylim(0, 95); axp.set_xlabel(r"time $t/\tau_0$")
a4.set_ylabel(r"deposit $D(t)$"); a5.tick_params(labelleft=False); a6.tick_params(labelleft=False)
for axx, L in zip(ax.ravel(), "abcdef"): panel(axx, L)
fig.tight_layout(h_pad=1.3, w_pad=1.2)
save(fig, "fig02_event_spectrum_6panel")


## Fig. 3 — fluctuations under removal

$m=c=\tau_0=1$, $\psi=0.2$, $R_\infty=5$. (a) One Lévy-driven OU path per law with mean and $\pm2\sigma(t)$ band; (b) exact $\gamma_1(t)$ with removal (Eq. 18) and without ($\propto t^{-1/2}$); (c) continuous removal and sloughing ($\eta=1/2$) driven by the same deposition; (d) stationary variance against $\eta$ at fixed $\psi$, Eq. (24), with Monte Carlo, for $c=1$ and $c=0.1$, and the continuous-removal baselines $s^2/2\psi$.

*Seeds `31`, `9`, `111` (one per sub-simulation: OU paths, sloughing, continuous-removal baseline). Runtime ~8 s. Saved to `../figures/fig03_removal.{pdf,png}`.*


In [ ]:
# ---- Main Fig. 3: fluctuations under removal ----
rng = np.random.default_rng(31)
LCOL = {"compound Poisson": "#0072B2", "Gamma": "#D55E00", "inverse Gaussian": "#009E73"}
LLS = {"compound Poisson": "-", "Gamma": "--", "inverse Gaussian": ":"}
m, c, tau0, psi = 1.0, 1.0, 1.0, 0.2
s2 = c**2 * m**2 * tau0; tc = c**2 * tau0; ell = m * tc; Rinf = m / psi
Q3 = {"compound Poisson": 1.5, "Gamma": 2.0, "inverse Gaussian": 3.0}   # q3/(l q2) = (2-a)/(1-a)
def incs(law, n, dt, r):
    if law == "Gamma": return r.gamma(dt / tc, ell, n)
    if law == "inverse Gaussian": return r.wald(m * dt, m * dt**2 / tc, n)
    k = r.poisson(2 * dt / tc, n); out = np.zeros(n); nz = k > 0; out[nz] = r.gamma(k[nz], ell / 2); return out

fig, ax = plt.subplots(2, 2, figsize=(COL2, 4.5))
(a1, a2), (a3, a4) = ax
# (a) one OU path per law
T, dt = 60.0, 0.02; tg = np.arange(0, T + dt / 2, dt)
mean = Rinf * (1 - np.exp(-psi * tg)); sd = np.sqrt(s2 / (2 * psi) * (1 - np.exp(-2 * psi * tg)))
a1.fill_between(tg, mean - 2 * sd, mean + 2 * sd, color=INK2, alpha=0.12, lw=0)
for law, seed in [("compound Poisson", 2), ("Gamma", 7), ("inverse Gaussian", 4)]:
    r = np.random.default_rng(seed); d = incs(law, len(tg) - 1, dt, r); R = np.zeros(len(tg))
    for i in range(1, len(tg)): R[i] = R[i - 1] * np.exp(-psi * dt) + d[i - 1]
    a1.plot(tg, R, color=LCOL[law], lw=0.9, label=law, alpha=0.95)   # noisy paths: solid, colour only
a1.plot(tg, mean, color=INK, lw=1.0, ls=(0, (4, 2)))
a1.set_xlim(0, T); a1.set_ylim(-0.5, 11.5)
a1.set_xlabel(r"time $t/\tau_0$"); a1.set_ylabel(r"$R(t)$")
a1.legend(loc="upper left", fontsize=7, ncol=3, handlelength=1.6, columnspacing=0.9)
# (b) exact skewness gamma_1(t) with and without removal
t = np.logspace(np.log10(0.3), np.log10(80), 300)
for law in LCOL:
    q3 = Q3[law] * ell * s2
    k2 = s2 * (1 - np.exp(-2 * psi * t)) / (2 * psi); k3 = q3 * (1 - np.exp(-3 * psi * t)) / (3 * psi)
    a2.loglog(t, k3 / k2**1.5, color=LCOL[law], ls="-")                       # with removal
    a2.loglog(t, q3 / (s2**1.5 * np.sqrt(t)), color=LCOL[law], ls=(0, (4, 2)))   # no removal
a2.axvline(1 / psi, color=INK2, lw=0.6, ls=":")
a2.text(1 / psi * 1.08, 0.17, r"$t=\psi^{-1}$", fontsize=7.5 * 1.15, color=INK2)
a2.set_xlim(0.3, 80); a2.set_ylim(0.15, 7)
from matplotlib.ticker import FixedLocator, NullFormatter, FormatStrFormatter
a2.yaxis.set_major_locator(FixedLocator([0.2, 0.5, 1, 2, 5])); a2.yaxis.set_major_formatter(FormatStrFormatter("%g")); a2.yaxis.set_minor_formatter(NullFormatter())
a2.xaxis.set_major_locator(FixedLocator([0.3, 1, 3, 10, 30])); a2.xaxis.set_major_formatter(FormatStrFormatter("%g")); a2.xaxis.set_minor_formatter(NullFormatter())
a2.set_xlabel(r"time $t/\tau_0$"); a2.set_ylabel(r"skewness $\gamma_1(t)$")
# (c) continuous removal vs sloughing, same deposition increments
eta = 0.5; rs = psi / eta; r = np.random.default_rng(9)
d = incs("Gamma", len(tg) - 1, dt, r); r.random(len(tg) - 1)        # keeps the other paths' draws unchanged
hits = np.random.default_rng(111).random(len(tg) - 1) < -np.expm1(-rs * dt)   # sloughing times
Rc = np.zeros(len(tg)); Rs = np.zeros(len(tg))
for i in range(1, len(tg)):
    Rc[i] = Rc[i - 1] * np.exp(-psi * dt) + d[i - 1]
    Rs[i] = (Rs[i - 1] + d[i - 1]) * ((1 - eta) if hits[i - 1] else 1.0)
# per-element detachment: each step's deposit is one element, removed whole after an Exp(psi) lifetime
life = r.exponential(1 / psi, len(d)); born = np.arange(1, len(tg))
dead = np.minimum(born + np.ceil(life / dt).astype(int), len(tg))
flux = np.zeros(len(tg) + 1); np.add.at(flux, born, d); np.add.at(flux, dead, -d)
Rp = np.cumsum(flux)[:len(tg)]
# independent-site blocking: R_inf/ell sites, each filled once at rate m/R_inf by an event of size ell
nsite = int(round(Rinf / ell)); tfill = np.sort(r.exponential(Rinf / m, nsite))
Rb = ell * np.searchsorted(tfill, tg, side="right")
CSL, CCO, CPE, CBL = "#CC79A7", "#E69F00", "#56B4E9", "#9a9893"   # distinct from the law colours of (a)-(b)
a3.plot(tg, Rs, color=CSL, lw=1.0, label=r"sloughing ($\eta=\frac{1}{2}$)")
a3.plot(tg, Rp, color=CPE, lw=1.0, label="per-element detachment")
a3.plot(tg, Rc, color=CCO, lw=1.0, label="continuous removal")
a3.plot(tg, Rb, color=CBL, lw=1.0, label="site blocking", zorder=1.5)
a3.plot(tg, mean, color=INK, lw=1.0, ls=(0, (4, 2)), zorder=5)
a3.set_xlim(0, T); a3.set_ylim(-0.5, 14.5); a3.set_yticks([0, 5, 10])
a3.set_xlabel(r"time $t/\tau_0$"); a3.set_ylabel(r"$R(t)$")
a3.legend(loc="upper left", fontsize=7, ncol=1, handlelength=1.6)
# (d) stationary variance against eta at fixed psi, Eq. (24), with Monte Carlo
etas = np.linspace(0.01, 1, 200)
def var_slough(e, s2_): return s2_ / ((2 - e) * psi) + e / (2 - e) * Rinf**2
def mc_var(e, cc, P=4000, T=150.0, t0=50.0, dt=0.02):
    rr = np.random.default_rng(int(1000 * e + 7 * cc)); tcc = cc**2 * tau0; R = np.zeros(P); vals = []
    rse = psi / e
    for i in range(int(T / dt)):
        R = R + rr.gamma(dt / tcc, m * tcc, P)
        R[rr.random(P) < -np.expm1(-rse * dt)] *= (1 - e)
        if (i + 1) * dt >= t0 and i % 25 == 0: vals.append(R.copy())
    return np.var(np.array(vals))
for cc, col in [(1.0, INK), (0.1, INK2)]:
    a4.semilogy(etas, var_slough(etas, cc**2 * m**2 * tau0), color=col)
    a4.axhline(cc**2 * m**2 * tau0 / (2 * psi), color=col, lw=0.8, ls=(0, (4, 2)))
    em = np.array([0.1, 0.25, 0.5, 0.75, 1.0])
    a4.plot(em, [mc_var(e, cc) for e in em], "o", color=col, ms=3.4, mfc="white", mew=0.8)
a4.text(0.98, 2.5 * 1.25, r"continuous, $c=1$", ha="right", fontsize=7, color=INK)
a4.text(0.98, 0.025 * 1.25, r"continuous, $c=0.1$", ha="right", fontsize=7, color=INK2)
a4.text(0.55, 20, r"sloughing, $c=1$", fontsize=7, color=INK, ha="right")
a4.text(0.06, 0.32, r"sloughing, $c=0.1$", fontsize=7, color=INK2, ha="left")
a4.set_xlim(0, 1.02); a4.set_ylim(1e-2, 60)
a4.set_xlabel(r"shed fraction $\eta$ (fixed $\psi=r_s\eta$)"); a4.set_ylabel(r"$\mathrm{Var}[R]_\infty$")
for axx, L in zip(ax.ravel(), "abcd"): panel(axx, L)
fig.tight_layout(h_pad=1.3, w_pad=2.0)
save(fig, "fig03_removal")      # main.tex includes figures/fig_removal


## Fig. 4 — observability is governed by the noise-to-signal ratio alone

Ported from `make_theory_figures.py` (that script's companion remaining-useful-life
section is dead code, superseded by the ridgeline figure above, and is not ported).
Attenuated skewness $\gamma_{1,k}/\gamma_1$ against the aggregation window
$k=\Delta/\tau_0$, for $\rho=0,10,10^2,10^3,10^4$. No simulation: a closed-form
curve family, so there is no seed and the cell runs in well under a second.

*Saved to `../figures/fig04_observability.{pdf,png}`.*


In [ ]:
# ---- Main Fig. 4: observability vs. the noise-to-signal ratio rho ----
# ported from make_theory_figures.py (fig_observability section only --
# that script's remaining-useful-life section is dead code and is not ported)

def screen_angle(ax, x0, y0, x1, y1):
    p0 = ax.transData.transform((x0, y0))
    p1 = ax.transData.transform((x1, y1))
    return np.degrees(np.arctan2(p1[1] - p0[1], p1[0] - p0[0]))

fig, axA = plt.subplots(figsize=(COL1, 2.5))

k = np.logspace(-0.5, 5.5, 600)
rhos = (0.0, 1e1, 1e2, 1e3, 1e4)
labels = [r"$\rho=0$" if r == 0 else rf"$\rho=10^{{{int(np.log10(r))}}}$" for r in rhos]
for rho in rhos:
    y = k / (k + rho) ** 1.5
    axA.loglog(k, y, color=INK, lw=1.2)
    if rho > 0:
        axA.plot(2 * rho, (2 * rho) / (3 * rho) ** 1.5, "o", color=INK, ms=4, zorder=5)
axA.set_xlabel(r"aggregation window $k=\Delta/\tau_0$ (periods)")
axA.set_ylabel(r"attenuated skewness, $\gamma_{1,k}/\gamma_1$")
axA.set_ylim(1e-5, 2)

threshold = 1e-2
axA.axhline(threshold, color=INK, lw=0.8, ls="--", zorder=3)
fig.tight_layout()

min_gap = 0.22  # minimum spacing between stacked labels, in decades
last_log_y = np.inf
for rho, lab in zip(rhos, labels):
    k0 = 50 if rho == 1e4 else 0.5
    k1 = k0 * 1.15
    y0 = k0 / (k0 + rho) ** 1.5
    y1 = k1 / (k1 + rho) ** 1.5
    ang = screen_angle(axA, k0, y0, k1, y1)
    if rho == 1e4:
        log_y = np.log10(y0)
    else:
        log_y = min(np.log10(y0), last_log_y - min_gap)
        last_log_y = log_y
    axA.annotate(lab, xy=(k0, 10 ** log_y), xytext=(0, 4), textcoords="offset points",
                 rotation=ang, rotation_mode="anchor", fontsize=8, color=INK,
                 ha="left", va="bottom", annotation_clip=False, clip_on=False)

for k0, rho, text, dy, va in ((40.0, 1e4, "noise masking", -4, "top"),
                               (40.0, 0.0, "aggregation masking", 4, "bottom")):
    k1 = k0 * 1.15
    y0 = k0 / (k0 + rho) ** 1.5
    y1 = k1 / (k1 + rho) ** 1.5
    ang = screen_angle(axA, k0, y0, k1, y1)
    axA.annotate(text, xy=(k0, y0), xytext=(0, dy), textcoords="offset points",
                 rotation=ang, rotation_mode="anchor", fontsize=8, color=INK,
                 style="italic", ha="left", va=va)

save(fig, "fig04_observability")


## Fig. 5 — distribution of the remaining useful life

The three laws at $c=2$, $2\times10^5$ paths per law, passage detected at integer sampling periods; (a) $n_x=223$, (b) $n_x=5$. Same simulation as the stored `fpt_validation_*.csv` statistics.

*Note: the code cell below reproduces the stored `fpt_validation_*.csv` statistics but is not the figure actually in the manuscript. The production Fig. 5 (`fig05_rul_ridgeline`) is generated in the "Extra" ridgeline section further down this notebook.*


In [ ]:
# ---- Main Fig. 5: distribution of the remaining useful life (two stacked panels) ----
LCOL = {"compound Poisson": "#0072B2", "Gamma": "#D55E00", "inverse Gaussian": "#009E73"}
LLS = {"compound Poisson": "-", "Gamma": "--", "inverse Gaussian": ":"}
m, tau0, c = 1.0, 1.0, 2.0
tc = c**2 * tau0; ell = m * tc
def period_incs(law, shape, r):
    if law == "Gamma": return r.gamma(tau0 / tc, ell, shape)
    if law == "inverse Gaussian": return r.wald(m * tau0, m * tau0**2 / tc, shape)
    k = r.poisson(2 * tau0 / tc, shape); out = np.zeros(shape); nz = k > 0; out[nz] = r.gamma(k[nz], ell / 2); return out
def passage_periods(law, nx, paths=200_000, chunk=20_000, seed=0):
    """First sampling period j at which D(j tau0) >= nx m tau0 (passage detected at the sampling instants)."""
    r = np.random.default_rng(seed); horizon = int(nx + 12 * c * np.sqrt(nx) + 40); out = []
    for _ in range(paths // chunk):
        D = np.cumsum(period_incs(law, (chunk, horizon), r), axis=1)
        hit = D >= nx * m * tau0
        out.append(np.argmax(hit, axis=1) + 1)
    return np.concatenate(out)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(COL1, 4.4))
for ax, nx, xr in [(a1, 223, (100, 360)), (a2, 5, (0, 30))]:
    for k, law in enumerate(LCOL):
        T = passage_periods(law, nx, seed=10 * nx + k)
        vals, cnt = np.unique(T, return_counts=True)
        pmf = np.zeros(xr[1] + 2); pmf[vals[vals <= xr[1] + 1]] = cnt[vals <= xr[1] + 1] / len(T)
        x = np.arange(len(pmf))
        ax.step(x, pmf, where="mid", color=LCOL[law], ls=LLS[law], lw=1.1, label=law)
    ax.axvline(nx, color=INK2, lw=0.8, ls=(0, (4, 2)))
    ax.axvline(nx + (1 + c**2) / 2, color=INK2, lw=0.8)
    ax.set_xlim(*xr); ax.set_ylim(bottom=0)
    ax.set_ylabel("probability mass")
    ax.text(0.97, 0.95, rf"$n_x={nx}$", transform=ax.transAxes, ha="right", va="top", fontsize=8.5)
a2.set_xlabel(r"first-passage time $T_x/\tau_0$ (periods)")
a2.legend(loc="center right", fontsize=7, handlelength=2.2)
a1.text(0.03, 0.95, "dashed: $n_x$\nsolid: $n_x+(1+c^2)/2$", transform=a1.transAxes, ha="left", va="top",
        fontsize=7, color=INK2)
panel(a1, "a"); panel(a2, "b")
fig.tight_layout(h_pad=1.2)
save(fig, "fig_rul_distribution")


# Supplemental Material

## Figure S1 — the tempered-stable family is a continuum (Sec. S3)

With $u=1/(1-\alpha)$, the squared relative width of the mass spectrum $w(x)$, the standardized cumulants are $\gamma_1/c=1+u$ and $\gamma_2/c^2=(1+u)(1+2u)$. Compound Poisson ($u=1/2$), Gamma ($u=1$) and inverse Gaussian ($u=2$) are points on these curves. Panel (b) shows $w(x)$, a Gamma density of shape $1-\alpha$ and mean $\ell$. Analytic, no simulation.

*Seed: none (analytic). Runtime negligible. Saved to `../figures/figS1_alpha_continuum.{pdf,png}`.*


In [ ]:
# ---- Figure S1: the tempered-stable family is a continuum ----
from scipy.special import gammaln

# with u = 1/(1-alpha) (squared relative width of the mass spectrum):
#   gamma1/c = 1+u,   gamma2/c^2 = (1+u)(1+2u)
def w(x_over_l, a):                       # mass spectrum Gamma(1-a, rate (1-a)/l), units of 1/l
    k = 1 - a
    return np.exp(k*np.log(k) + (k-1)*np.log(x_over_l) - k*x_over_l - gammaln(k))

fig, (axa, axb) = plt.subplots(1, 2, figsize=(COL2, 2.7))
u = np.linspace(0, 4, 400)
axa.semilogy(u, 1 + u, color=INK)
axa.semilogy(u, (1 + u) * (1 + 2*u), color=INK, ls="--")
for name, al in LAWS.items():
    uu = 1 / (1 - al)
    axa.plot([uu, uu], [1 + uu, (1 + uu)*(1 + 2*uu)], "o", color=INK, ms=4.5, zorder=5)
    axa.plot([uu, uu], [1 + uu, (1 + uu)*(1 + 2*uu)], color=INK2, lw=0.6, ls=":", zorder=1)
    short = {"compound Poisson": "CP", "Gamma": "Gamma", "inverse Gaussian": "IG"}[name]
    axa.text(uu - 0.09, (1 + uu)*(1 + 2*uu), short, ha="right", va="center", fontsize=7.5, color=INK2)
inline(axa, 2.6, 4.0, r"$\gamma_1/c=1+u$", rot=7)
inline(axa, 2.45, 33, r"$\gamma_2/c^2=(1+u)(1+2u)$", rot=9)
inline(axa, 0.06, 0.88, r"$u\to0$: equal events", fontsize=7.5, color=INK2, va="top")
axa.set_xlim(0, 4); axa.set_ylim(0.6, 80)
axa.set_xlabel(r"$u=1/(1-\alpha)$, squared relative width of $w(x)$")
axa.set_ylabel(r"$\gamma_1/c$ (solid), $\gamma_2/c^2$ (dashed)")
sec = axa.secondary_xaxis("top", functions=(lambda v: 1 - 1/np.maximum(v, 1e-9), lambda a: 1/(1 - a)))
sec.set_xticks([-1, 0, 0.5, 0.75]); sec.set_xticklabels(["$-1$", "$0$", "$1/2$", "$3/4$"])
sec.set_xlabel(r"shape parameter $\alpha$", labelpad=3)
sec.minorticks_off()
axa.text(0.0, 1.20, "(a)", transform=axa.transAxes, ha="left", va="bottom", fontsize=9)

x = np.logspace(-3, np.log10(6), 600)
styles = [(-4, ":", INK2), (-1, "-.", INK2), (0, "-", INK), (0.5, "--", INK), (0.8, (0, (1, 1)), INK)]
for al, ls, cl in styles:
    axb.loglog(x, w(x, al), color=cl, ls=ls)
for al, xx, yy in [(0.8, 1.3e-3, 70), (0.5, 1.3e-3, 4.0), (0, 1.3e-3, 1.25), (-1, 2.0e-2, 0.03), (-4, 0.012, 2.2e-3)]:
    inline(axb, xx, yy, rf"$\alpha={al:g}$", fontsize=7.5, va="center")
axb.axvline(1, color=INK2, lw=0.6, ls=":")
inline(axb, 1.12, 25, r"mean $\ell$", fontsize=7.5, color=INK2, va="center")
axb.set_xlim(1e-3, 6); axb.set_ylim(1e-3, 150)
axb.set_xlabel(r"event size $x/\ell$")
axb.set_ylabel(r"mass spectrum $\ell\,w(x)$")
panel(axb, "b")
fig.tight_layout(w_pad=2.5)
save(fig, "figS1_alpha_continuum")


## Figure S2 — what removal does to the moments (Secs. IV and V C, S4)

(a) Transient variance for continuous linear removal (Eq. 18), per-element detachment ($q_2(1-e^{-\psi t})/\psi$, Sec. S4) and Poisson-timed sloughing (Eqs. S16–S17, solved numerically), with Monte Carlo for Gamma deposition. (b) Stationary autocorrelation, estimated across paths. (c) Independent-site Langmuir blocking against continuous removal at the same $R_\infty$ and $\psi$, monodisperse events.

Parameters as Fig. 3 of the main text: $m=c=\tau_0=1$, $\psi=0.2$, $\eta=1/2$. Per-element detachment is simulated exactly, by Gamma–Beta thinning of the alive deposit (see the cell), so it carries no time-step bias; continuous removal and sloughing are stepped at $dt=0.005$, a relative bias of order $\psi\,dt=10^{-3}$.

*Seed `1`. Saved to `../figures/figS2_removal_moments.{pdf,png}`.*


In [ ]:
# ---- Figure S2: what removal does to the moments ----
from scipy.integrate import solve_ivp
rng = np.random.default_rng(1)

m, c, tau0, psi = 1.0, 1.0, 1.0, 0.2           # as Fig. 3 of the main text
s2 = c**2 * m**2 * tau0; tc = s2 / m**2; ell = m * tc; Rinf = m / psi
eta = 0.5; rs = psi / eta
base = s2 / (2 * psi)                          # continuous-removal plateau variance

# ---------- analytic curves ----------
t = np.linspace(0, 40, 400)
v_cont = base * (1 - np.exp(-2 * psi * t))             # Eq. (18), n = 2
v_elem = (s2 / psi) * (1 - np.exp(-psi * t))           # per-element detachment (Sec. S4)
def slough_rhs(_, y):                                  # Eqs. (S16)-(S17)
    m1, m2 = y
    return [m - rs*eta*m1, 2*m*m1 + s2 - rs*eta*(2 - eta)*m2]
sol = solve_ivp(slough_rhs, (0, 40), [0, 0], t_eval=t, rtol=1e-10, atol=1e-12)
v_slou = sol.y[1] - sol.y[0]**2

# ---------- Monte Carlo, Gamma deposition ----------
# Per-element detachment is simulated exactly, with no time-step bias. For Gamma
# deposition the deposit alive at time t is a Poisson cloud of jumps with Levy
# measure A(t) x^-1 e^{-lam x} dx, i.e. Gamma(A(t), lam) distributed. Over a step h
# each element survives independently with p = e^{-psi h}; independent thinning of
# a Gamma(A) cloud keeps the fraction Beta(pA, (1-p)A) of its mass, and the
# deposit made during the step that is still alive at its end is
# Gamma(B(1-p)/psi, lam). Hence A_{k+1} = p A_k + B(1-p)/psi exactly.
lam_g = 1.0 / ell; B_g = m * lam_g                 # Gamma law: alpha = 0, Eq. (S7)
def simulate(dt=0.005, T=120.0, P=20000, every=10):
    """Gamma deposition on P paths: continuous removal, sloughing, per-element detachment."""
    n = int(round(T / dt)); out = {k: [] for k in "ces"}; times = []
    Rc = np.zeros(P); Rs = np.zeros(P); Re = np.zeros(P); A = 0.0
    p = np.exp(-psi * dt); a_new = B_g * (1 - p) / psi
    for i in range(n):
        ti = (i + 1) * dt
        dD = rng.gamma(dt / tc, ell, size=P)
        Rc = Rc * np.exp(-psi * dt) + dD                # continuous linear removal
        Rs = Rs + dD
        Rs[rng.random(P) < -np.expm1(-rs * dt)] *= (1 - eta)   # Poisson-timed sloughing
        keep = rng.beta(p * A, (1 - p) * A, size=P) if A > 0 else np.zeros(P)
        Re = Re * keep + rng.gamma(a_new, 1 / lam_g, size=P)   # per-element detachment, exact
        A = p * A + a_new
        if (i + 1) % every == 0:
            out["c"].append(Rc.copy()); out["s"].append(Rs.copy()); out["e"].append(Re.copy())
            times.append(ti)
    return np.array(times), {k: np.array(v) for k, v in out.items()}

ts, X = simulate()

def acf(Y, ts, t1s=np.arange(60.0, 90.1, 2.0), lags=np.arange(0, 21, 2)):
    """Ensemble estimate: correlation across paths of R(t1) and R(t1+lag), averaged over t1."""
    idx = lambda tt: int(np.argmin(np.abs(ts - tt)))
    r = []
    for L in lags:
        r.append(np.mean([np.corrcoef(Y[idx(t1)], Y[idx(t1 + L)])[0, 1] for t1 in t1s]))
    return lags, np.array(r)

# ---------- blocking vs removal (monodisperse events, l = J1) ----------
J1 = ell; Ns = int(round(Rinf / J1))
tb = np.linspace(0, 40, 400); theta = 1 - np.exp(-psi * tb)
v_block = J1 * Rinf * theta * (1 - theta)              # independent-site filling (Sec. V C)
v_cont_mono = (m * J1 / (2 * psi)) * (1 - np.exp(-2 * psi * tb))   # same R_inf, psi, s^2 = m J1
fill = rng.exponential(1 / psi, size=(20000, Ns))
tb_mc = np.arange(2, 40, 2.0)
vb_mc = np.array([(J1 * (fill <= tt).sum(1)).var() for tt in tb_mc])

# ---------- figure ----------
fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(COL2, 2.45))
mstep = max(1, int(round(1.2 / (ts[1] - ts[0]))))   # one marker per 1.2 time units
a1.plot(t, v_slou / base, color=INK, ls="--")
a1.plot(t, v_elem / base, color=INK, ls="-.")
a1.plot(t, v_cont / base, color=INK)
for k, mk in [("s", "s"), ("e", "^"), ("c", "o")]:
    a1.plot(ts[::mstep], X[k].var(1)[::mstep] / base, mk, color=INK, ms=3.2, mfc="white", mew=0.8)
inline(a1, 22, v_slou[-1] / base + 0.15, "sloughing", fontsize=7.5)
inline(a1, 22, v_elem[-1] / base + 0.15, "per-element", fontsize=7.5)
inline(a1, 22, v_cont[-1] / base + 0.15, "continuous", fontsize=7.5)
a1.set_xlim(0, 40); a1.set_ylim(0, 5.4)
a1.set_xlabel(r"time $t/\tau_0$"); a1.set_ylabel(r"$\mathrm{Var}[R(t)]\;/\;(s^2/2\psi)$")
panel(a1, "a")

tau = np.linspace(0, 21, 200)
a2.semilogy(tau, np.exp(-psi * tau), color=INK)
for k, mk in [("c", "o"), ("e", "^"), ("s", "s")]:
    L, r = acf(X[k], ts)
    a2.plot(L + {"c": 0, "e": 0.35, "s": 0.7}[k], r, mk, color=INK, ms=3.2, mfc="white", mew=0.8)
inline(a2, 8, 0.32, r"$e^{-\psi\tau}$", fontsize=9.75)
a2.set_xlim(0, 21); a2.set_ylim(1e-2, 1.3)
a2.set_xlabel(r"lag $\tau/\tau_0$"); a2.set_ylabel("stationary autocorrelation")
panel(a2, "b")

a3.plot(psi * tb, v_cont_mono / (J1 * Rinf), color=INK)
a3.plot(psi * tb, v_block / (J1 * Rinf), color=INK, ls="--")
a3.plot(psi * tb_mc, vb_mc / (J1 * Rinf), "o", color=INK, ms=3.2, mfc="white", mew=0.8)
a3.axvline(np.log(2), color=INK2, lw=0.6, ls=":")
inline(a3, 4.0, 0.515, "removal", fontsize=7.5, ha="right")
inline(a3, 2.6, 0.12, "blocking", fontsize=7.5, rot=-28)
inline(a3, np.log(2) + 0.1, 0.02, r"$\theta=\frac{1}{2}$", fontsize=7.5, color=INK2)
a3.set_xlim(0, 8); a3.set_ylim(0, 0.6)
a3.set_xlabel(r"relaxation times $\psi t$"); a3.set_ylabel(r"$\mathrm{Var}[R(t)]\;/\;(\ell R_\infty)$")
panel(a3, "c")
fig.tight_layout(w_pad=1.6)
save(fig, "figS2_removal_moments")
st = ts >= 60
print({k: round(float(X[k][st].var()), 3) for k in X}, "theory", base, s2/psi, v_slou[-1])


## Figure S5 — Monte Carlo validation of the record-length estimate (Sec. V D, S6)

Records of $N$ base periods are generated for the Gamma and inverse-Gaussian laws at $c=2$, observed with Gaussian noise of variance set by $\rho$ (Eq. 29), differenced over disjoint windows of $k=\rho/2$ periods ($n_w=N/2k$ windows). The law is assigned by comparing the sample skewness with the midpoint of the two attenuated values of Eq. (30). With $d'=2\sqrt{N/N^\ast}$, the asymptotic prediction is $P=\Phi(\sqrt{N/N^\ast})$, equal to $\Phi(1)=0.84$ at $N=N^\ast=324\rho^2/c^2$. 1000 records per law and point.

*Seed `7`. Runtime ~45 s (1000 records per law per point, 4 values of $\rho$). Saved to `../figures/figS5_record_validation.{pdf,png}`; also caches `../data/figC_mc.npz`.*


In [ ]:
# ---- Figure S5: Monte Carlo validation of the record-length estimate, Eq. (31) ----
from scipy.stats import norm, skew
rng = np.random.default_rng(7)

m, tau0, c = 1.0, 1.0, 2.0
tc = c**2 * tau0; ell = m * tc
g1 = {"Gamma": 2 * c, "IG": 3 * c}                       # increment skewness at one period (Table II)

def window_increments(law, k, n, sigma):
    """n disjoint differenced increments over k periods: Delta D_k + eps_a - eps_b."""
    D = k * tau0
    if law == "Gamma":
        x = rng.gamma(D / tc, ell, size=n)
    else:
        x = rng.wald(m * D, m * D**2 / tc, size=n)       # IG(mean m D, shape m D^2/tc)
    return x + sigma * (rng.standard_normal(n) - rng.standard_normal(n))

def p_correct(rho, N_over_Nstar, reps=1000):
    sigma = np.sqrt(rho * c**2 * m**2 * tau0**2 / 2)    # Eq. (29)
    k = max(1, int(round(rho / 2)))                     # detection-optimal window
    Nstar = 324 * rho**2 / c**2                         # Eq. (31)
    exp1 = {L: g1[L] * k / (k + rho)**1.5 for L in g1}  # attenuated skewness, Eq. (30)
    thr = 0.5 * (exp1["Gamma"] + exp1["IG"])            # midpoint rule
    out = []
    for f in N_over_Nstar:
        nw = max(8, int(f * Nstar / (2 * k)))
        ok = 0
        for _ in range(reps):
            ok += skew(window_increments("Gamma", k, nw, sigma)) < thr
            ok += skew(window_increments("IG", k, nw, sigma)) >= thr
        out.append(ok / (2 * reps))
    return np.array(out)

fN = np.array([0.03, 0.1, 0.25, 0.5, 1.0, 2.0, 4.0])
rhos = [10, 40, 160, 640]
res = {r: p_correct(r, fN) for r in rhos}
np.savez(DATA / "figC_mc.npz", fN=fN, **{f"r{r}": v for r, v in res.items()})

fig, ax = plt.subplots(figsize=(COL1, 2.55))
ff = np.logspace(np.log10(0.02), np.log10(6), 200)
ax.semilogx(ff, norm.cdf(np.sqrt(ff)), color=INK)           # d' = 2 sqrt(N/N*), P = Phi(d'/2)
mk = {10: "s", 40: "^", 160: "D", 640: "o"}
for r in rhos:
    ax.plot(fN, res[r], mk[r], color=INK, ms=3.6, mfc="white", mew=0.8, label=rf"$\rho={r}$")
ax.axvline(1, color=INK2, lw=0.6, ls=":"); ax.axhline(norm.cdf(1), color=INK2, lw=0.6, ls="--")
inline(ax, 0.93, 1.0, r"$N^\ast=324\,\rho^2/c^2$", fontsize=7.5, color=INK2, va="top", ha="right")
inline(ax, 0.022, norm.cdf(1) + 0.012, r"$d'=2$: $P=\Phi(1)$", fontsize=7.5, color=INK2)
inline(ax, 0.07, 0.66, r"$\Phi(\sqrt{N/N^\ast})$", rot=19, fontsize=8)
ax.set_xlim(0.02, 6); ax.set_ylim(0.45, 1.01)
ax.set_xlabel(r"record length $N/N^\ast$")
ax.set_ylabel("probability of correct law")
ax.legend(loc="lower right", handletextpad=0.2, fontsize=7)
save(fig, "figS5_record_validation")
for r in rhos: print(r, np.round(res[r], 3))


## Figure S3 — nonlinear removal breaks the universality of the mean (Sec. IV C, S4)

$g(R)=\psi R+\varepsilon R^2$ with $m=\tau_0=1$, $\psi=0.2$, $\varepsilon=0.05$, Gamma deposition. (a) Plateau shift from the mean-field value against $1/N_\infty=\psi t_c$ (varying $c$): second-order closure, Eq. (26), and Monte Carlo; linear removal ($\varepsilon=0$) shows no shift. (b) Mean trajectory at $c=1$. Euler scheme, $dt=0.01$, 4000 paths.

*Seed `11`. Runtime ~11 s (Euler scheme, 4000 paths). Saved to `../figures/figS3_nonlinear_removal.{pdf,png}`.*


In [ ]:
# ---- Figure S3: nonlinear removal breaks the universality of the mean ----
from scipy.optimize import brentq
from scipy.integrate import solve_ivp
rng = np.random.default_rng(11)

m, tau0, psi, eps = 1.0, 1.0, 0.2, 0.05            # g(R) = psi R + eps R^2
g   = lambda R: psi * R + eps * R**2
gp  = lambda R: psi + 2 * eps * R
gpp = 2 * eps

mu_mf = brentq(lambda x: m - g(x), 0, 100)         # mean-field plateau
def closure_plateau(c):                             # Eq. (26) at stationarity
    s2 = c**2 * m**2 * tau0
    f = lambda mu: m - g(mu) - 0.5 * gpp * s2 / (2 * gp(mu))
    return brentq(f, 0, mu_mf)

def mc_plateau(c, e=eps, dt=0.01, T=80.0, t0=40.0, P=4000):
    tc = c**2 * tau0; ell = m * tc
    R = np.zeros(P); acc = 0.0; cnt = 0
    for i in range(int(T / dt)):
        R = np.maximum(R + rng.gamma(dt / tc, ell, size=P) - (psi * R + e * R**2) * dt, 0.0)
        if (i + 1) * dt >= t0 and i % 10 == 0:
            acc += R.mean(); cnt += 1
    return acc / cnt

cs = np.array([0.25, 0.5, 1.0, 1.5, 2.0, 2.5])
mc_nl = np.array([mc_plateau(c) for c in cs])
mc_lin = np.array([mc_plateau(c, e=0.0) for c in cs])
cgrid = np.linspace(0.05, 2.6, 120)
cl = np.array([closure_plateau(c) for c in cgrid])
x_cl = cgrid**2 * psi * tau0          # 1/N_inf = psi t_c

# mean trajectories at c = 2 (panel b)
c_b = 1.0; s2b = c_b**2 * m**2 * tau0
t = np.linspace(0, 40, 400)
mf_traj = solve_ivp(lambda _, y: [m - g(y[0])], (0, 40), [0.0], t_eval=t).y[0]
def clos_rhs(_, y):
    mu, V = y
    return [m - g(mu) - 0.5 * gpp * V, s2b - 2 * gp(mu) * V]
cl_traj = solve_ivp(clos_rhs, (0, 40), [0.0, 0.0], t_eval=t).y[0]
dt, P = 0.01, 4000; tcb = c_b**2 * tau0; R = np.zeros(P); tm, mm = [], []
for i in range(int(40 / dt)):
    R = np.maximum(R + rng.gamma(dt / tcb, m * tcb, size=P) - g(R) * dt, 0.0)
    if (i + 1) % 100 == 0: tm.append((i + 1) * dt); mm.append(R.mean())

fig, (a1, a2) = plt.subplots(1, 2, figsize=(COL2, 2.6))
# x-axis: relative event size l/mu, one size-biased event against the plateau deposit,
# each series at its own plateau (the closure is an expansion in this ratio)
ell_c = cs**2 * m * tau0                                 # l = m t_c = c^2 m tau0
x_cl = (cgrid**2 * m * tau0) / cl
x_mc = ell_c / mc_nl
x_lin = ell_c / mc_lin
a1.axhline(0, color=INK2, lw=0.6, ls=":")
a1.semilogx(x_cl, (cl - mu_mf) / mu_mf * 100, color=INK)
a1.plot(x_mc, (mc_nl - mu_mf) / mu_mf * 100, "o", color=INK, ms=3.8, mfc="white", mew=0.8)
a1.plot(x_lin, (mc_lin - m / psi) / (m / psi) * 100, "s", color=INK2, ms=3.4, mfc="white", mew=0.8)
inline(a1, 0.3, 0.9, r"linear removal, $\varepsilon=0$", fontsize=7.5, color=INK2)
a1.set_xlabel(r"relative event size $\ell/\mu$ (one event / plateau deposit)")
a1.set_ylabel(r"plateau shift from mean field (%)")
a1.set_xlim(0.01, 4); a1.set_ylim(-32, 2.5)
panel(a1, "a")

a2.plot(t, mf_traj, color=INK2, ls="--")
a2.plot(t, cl_traj, color=INK)
a2.plot(tm[::3], mm[::3], "o", color=INK, ms=3.2, mfc="white", mew=0.8)
inline(a2, 17, mu_mf + 0.08, r"mean field, $\dot\mu=m-g(\mu)$", fontsize=7.5, color=INK2)
inline(a2, 1.0, 3.05, r"$c=1$", fontsize=8)
a2.set_xlim(0, 40); a2.set_ylim(0, 3.3)
a2.set_xlabel(r"time $t/\tau_0$"); a2.set_ylabel(r"mean fouling $\mathbb{E}[R(t)]$")
panel(a2, "b")
fig.tight_layout(w_pad=2.5)
save(fig, "figS3_nonlinear_removal")
print("mu_mf", round(mu_mf, 3), "closure c=1", round(closure_plateau(1.0), 3))
print("MC nl", np.round(mc_nl, 3)); print("MC lin", np.round(mc_lin, 3))


## Figure S4 — finite-record convergence of the noise-free estimators (Sec. S6)

Sample skewness and excess kurtosis of one-period increments for the three laws at $c=2$, noise-free: median (markers) and 5–95 % band over 300 records per record length; dashed lines are the exact values of Table II of the main text ($\gamma_1=3,4,6$; $\gamma_2=12,24,60$). Law colours: Okabe–Ito palette (colour-blind safe).

*Seed `5`. Runtime ~2 s. Saved to `../figures/figS4_mechanism_recovery.{pdf,png}`.*


In [ ]:
# ---- Figure S4: finite-record convergence of the noise-free estimators (Sec. S6) ----
from scipy.stats import skew as sskew, kurtosis as skurt
rng = np.random.default_rng(5)
LCOL = {"compound Poisson": "#0072B2", "Gamma": "#D55E00", "inverse Gaussian": "#009E73"}  # Okabe-Ito, colour-blind safe
LLS = {"compound Poisson": "-", "Gamma": "--", "inverse Gaussian": ":"}

m, tau0, c = 1.0, 1.0, 2.0
tc = c**2 * tau0; ell = m * tc
def increments(law, n):                     # one-period increments, Table II of the main text
    if law == "Gamma":
        return rng.gamma(tau0 / tc, ell, size=n)
    if law == "inverse Gaussian":
        return rng.wald(m * tau0, m * tau0**2 / tc, size=n)
    k = rng.poisson(2 * tau0 / tc, size=n)  # compound Poisson: rate 2/tc, Exp(mean l/2) jumps
    out = np.zeros(n); nz = k > 0
    out[nz] = rng.gamma(k[nz], ell / 2)
    return out
g1 = {"compound Poisson": 1.5 * c, "Gamma": 2 * c, "inverse Gaussian": 3 * c}
g2 = {"compound Poisson": 3 * c**2, "Gamma": 6 * c**2, "inverse Gaussian": 15 * c**2}

Ns = np.unique(np.round(np.logspace(2, 4, 9)).astype(int)); R = 300
stats_ = {}
for law in LCOL:
    sk = np.empty((len(Ns), R)); ku = np.empty_like(sk)
    for i, N in enumerate(Ns):
        X = increments(law, N * R).reshape(R, N)
        sk[i] = sskew(X, axis=1); ku[i] = skurt(X, axis=1)
    stats_[law] = (sk, ku)

fig, (a1, a2) = plt.subplots(2, 1, figsize=(COL1, 4.6), sharex=True)
for ax, idx, th, lab in [(a1, 0, g1, r"sample skewness"), (a2, 1, g2, r"sample excess kurtosis")]:
    for law, col in LCOL.items():
        Y = stats_[law][idx]
        med, lo, hi = np.median(Y, 1), np.quantile(Y, 0.05, 1), np.quantile(Y, 0.95, 1)
        ax.fill_between(Ns, lo, hi, color=col, alpha=0.15, lw=0)
        ax.plot(Ns, med, "o", ls=LLS[law], color=col, ms=3, lw=1.1, label=law)
        ax.axhline(th[law], color=col, lw=0.8, ls=(0, (4, 2)))
    ax.set_xscale("log"); ax.set_ylabel(lab)
a1.set_ylim(0, 11.5); a2.set_ylim(0, 115)
a2.set_xlabel(r"record length $N$ (independent periods)")
a1.legend(loc="upper left", fontsize=7, handlelength=2.2, ncol=3, columnspacing=1.0, handletextpad=0.4)
panel(a1, "a"); panel(a2, "b")
fig.tight_layout(h_pad=1.2)
save(fig, "figS4_mechanism_recovery")
for law in LCOL:
    print(law, "median skew @N=1e4:", round(float(np.median(stats_[law][0][-1])), 2), "theory", g1[law],
          "| kurt:", round(float(np.median(stats_[law][1][-1])), 1), "theory", g2[law])


## Figure S6 — range of validity of the first-passage formulas (Sec. S7)

Redrawn from the stored Monte Carlo summaries `fpt_validation_mean.csv` and `fpt_validation_shape.csv` (in `data/`; $c=2$, $n_x\in\{5,10,25,50,223\}$). (a) $\mathbb{E}[T_x]-n_x$ with ±1 Monte Carlo standard error against $(1+c^2)/2$; (b) $\mathrm{sd}[T_x]/\sqrt{n_x}$ against $c$; (c) $\mathrm{skew}[T_x]\sqrt{n_x}$ against $3c-\gamma_1$.

*Seed: none (reads stored CSVs, no simulation). Runtime ~2 s. Saved to `../figures/figS6_fpt_validation.{pdf,png}`.*


In [ ]:
# ---- Figure S6: range of validity of the first-passage formulas (Sec. S7) ----
import csv
LCOL = {"compound Poisson": "#0072B2", "Gamma": "#D55E00", "inverse Gaussian": "#009E73"}  # Okabe-Ito, colour-blind safe
LLS = {"compound Poisson": "-", "Gamma": "--", "inverse Gaussian": ":"}
c = 2.0
G1 = {"compound Poisson": 1.5 * c, "Gamma": 2 * c, "inverse Gaussian": 3 * c}
def read(fn):
    with open(fn) as f: return list(csv.DictReader(f))
mean_rows = read(DATA / "fpt_validation_mean.csv"); shape_rows = read(DATA / "fpt_validation_shape.csv")

fig, (a1, a2, a3) = plt.subplots(1, 3, figsize=(COL2, 2.35))
for law, col in LCOL.items():
    mr = [r for r in mean_rows if r["law"] == law]; sr = [r for r in shape_rows if r["law"] == law]
    nx = np.array([float(r["n_x"]) for r in mr])
    err = np.array([float(r["E_T"]) - float(r["n_x"]) for r in mr])
    se = np.array([abs(float(r["error"]) / float(r["err_in_SE"])) for r in mr])
    a1.errorbar(nx, err, yerr=se, fmt="o", ls=LLS[law], color=col, ms=3, lw=1.1, capsize=1.5, label=law)
    nxs = np.array([float(r["n_x"]) for r in sr])
    a2.plot(nxs, np.array([float(r["sd"]) for r in sr]) / np.sqrt(nxs), "o", ls=LLS[law], color=col, ms=3, lw=1.1)
    a3.plot(nxs, np.array([float(r["skew"]) for r in sr]) * np.sqrt(nxs), "o", ls=LLS[law], color=col, ms=3, lw=1.1)
    a3.axhline(3 * c - G1[law], color=col, lw=0.8, ls=(0, (4, 2)))
a1.axhline((1 + c**2) / 2, color=INK2, lw=0.8, ls=(0, (4, 2)))
a2.axhline(c, color=INK2, lw=0.8, ls=(0, (4, 2)))
a1.text(0.03, 0.97, r"dashed: $(1+c^2)/2$", transform=a1.transAxes, ha="left", va="top", fontsize=7.5, color=INK2)
a2.text(0.97, 0.03, r"dashed: $c$", transform=a2.transAxes, ha="right", va="bottom", fontsize=7.5, color=INK2)
a3.text(0.97, 0.42, r"dashed: $3c-\gamma_1$", transform=a3.transAxes, ha="right", va="center", fontsize=7.5, color=INK2)
for ax, lab in [(a1, r"$\mathbb{E}[T_x]-n_x$"), (a2, r"$\mathrm{sd}[T_x]/\sqrt{n_x}$"), (a3, r"$\mathrm{skew}[T_x]\,\sqrt{n_x}$")]:
    ax.set_xscale("log"); ax.set_xlabel(r"threshold distance $n_x$"); ax.set_ylabel(lab)
    ax.set_xlim(4, 300)
a1.set_ylim(2.2, 2.6); a2.set_ylim(1.6, 2.25); a3.set_ylim(-0.2, 3.6)
a1.legend(loc="lower right", fontsize=7, handlelength=2.2)
panel(a1, "a"); panel(a2, "b"); panel(a3, "c")
fig.tight_layout(w_pad=1.6)
save(fig, "figS6_fpt_validation")


# Fig. 5 (and a companion layout) — ridgeline of the remaining-life distribution

Compound Poisson at $c=2$: probability mass of $T_x/\tau_0$ (passage detected at integer sampling periods), from $10^5$ paths; all thresholds are read off the same paths. Each ridge is scaled to a common peak height and sits on a baseline at its $n_x$; the black tick marks the mean life $n_x+(1+c^2)/2$, Eq. (36). Two layouts: (A) one ridge every 20 periods of threshold distance, `fig05_rul_ridgeline` — **this is the manuscript's Fig. 5** (`fig:rul_distribution`); (B) log-spaced thresholds $n_x=5,10,\dots,160$ on evenly spaced ridges, `fig_rul_ridgeline_log`, which shows best how the right skew $(3c-\gamma_1)/\sqrt{n_x}$ fades as the threshold recedes — this second layout is **not** included in the manuscript.

*Seed `55`. Runtime ~4 s for both layouts together. Saved to `../figures/fig05_rul_ridgeline.{pdf,png}` and `../figures/fig_rul_ridgeline_log.{pdf,png}`.*

In [ ]:
# ---- Ridgeline (joyplot): remaining-life distributions for one law ----
# Compound Poisson only; all thresholds read off the same simulated paths. Each ridge is the
# probability mass of T_x/tau0, scaled to a common peak height, on a baseline at its own n_x.
LAW = "compound Poisson"                                 # greyscale figure
m, tau0, c = 1.0, 1.0, 2.0
tc = c**2 * tau0; ell = m * tc
nx_all = np.array(sorted(set(range(5, 201, 5)) | {160}))
rng = np.random.default_rng(55)
paths, chunk, horizon = 100_000, 10_000, 420             # horizon ~7 sd beyond the mean at n_x = 200
Tall = {nx: [] for nx in nx_all}
for _ in range(paths // chunk):
    k = rng.poisson(2 * tau0 / tc, (chunk, horizon)); inc = np.zeros(k.shape); nz = k > 0
    inc[nz] = rng.gamma(k[nz], ell / 2)                   # compound Poisson: Exp(mean l/2) jumps
    D = np.cumsum(inc, axis=1)
    for nx in nx_all:                                     # first period with D(j tau0) >= nx m tau0
        hit = D >= nx * m * tau0
        assert hit[:, -1].all(), "horizon too short"
        Tall[nx].append(np.argmax(hit, axis=1) + 1)
Tall = {nx: np.concatenate(v) for nx, v in Tall.items()}

def ridgeline(nxs, ypos, amp, ylabel, yticks, yticklabels, fname, height, xmax):
    fig, ax = plt.subplots(figsize=(COL1, height)); ax.grid(False)
    for nx, y0 in sorted(zip(nxs, ypos), key=lambda p: -p[1]):           # back to front
        vals, cnt = np.unique(Tall[nx], return_counts=True)
        x = np.arange(0, horizon + 1); pmf = np.zeros(len(x)); pmf[vals] = cnt / len(Tall[nx])
        sup = np.nonzero(pmf > 2e-3 * pmf.max())[0]; lo, hi = max(sup[0] - 1, 0), min(sup[-1] + 2, horizon)
        x, pmf = x[lo:hi], pmf[lo:hi]; y = y0 + amp * pmf / pmf.max(); z = 300 - y0
        ax.fill_between(x, y0, y, step="mid", color="white", zorder=z)
        ax.fill_between(x, y0, y, step="mid", color="#d9d8d4", lw=0, zorder=z)
        ax.step(x, y, where="mid", color=INK2, lw=0.8, zorder=z + 0.5)
        ax.plot([x[0], x[-1]], [y0, y0], color=INK2, lw=0.5, zorder=z + 0.5)
        ax.plot([nx + (1 + c**2) / 2] * 2, [y0, y0 + amp], color=INK, lw=0.8, zorder=z + 0.6)   # mean life
    ax.set_yticks(yticks); ax.set_yticklabels(yticklabels); ax.yaxis.set_minor_locator(plt.NullLocator())
    ax.set_xlim(0, xmax); ax.set_ylim(min(ypos) - 0.02 * (max(ypos) - min(ypos)), max(ypos) + amp * 1.08)
    ax.set_xlabel(r"first-passage time $T_x/\tau_0$ (periods)"); ax.set_ylabel(ylabel)
    ax.text(0.03, 0.98, f"{LAW}, $c={c:g}$", transform=ax.transAxes, ha="left", va="top", fontsize=7.5, color=INK)
    ax.text(0.97, 0.03, r"tick: mean life $n_x+(1+c^2)/2$", transform=ax.transAxes, ha="right", va="bottom",
            fontsize=7, color=INK)
    save(fig, fname); return fig

# (A) one ridge every 20 periods of threshold distance
nA = [5] + list(range(20, 201, 20))
ridgeline(nA, nA, 30.0, r"threshold distance $n_x$", [5, 50, 100, 150, 200], ["5", "50", "100", "150", "200"],
          "fig05_rul_ridgeline", 3.3, 280)
# (B) log-spaced thresholds, evenly spaced ridges
nB = [5, 10, 20, 40, 80, 160]
ridgeline(nB, list(range(len(nB))), 1.5, r"threshold distance $n_x$", list(range(len(nB))), [str(n) for n in nB],
          "fig_rul_ridgeline_log", 3.0, 230)
